# S6E10 — GPU retrain + Save Version (LGB / CAT / TabM)

**Goal:** regenerate three **separate clean members**, with the shared `folds.csv`, internal ES (never outer fold), and export independent OOF / test predictions. This notebook does NOT stack or submit.

**GPU policy:** **CatBoost GPU 0**, **TabM CUDA GPU 1** if T4×2 is available (otherwise GPU 0), **LightGBM GPU/OpenCL** if functional; otherwise it prints a warning and falls back to CPU. LightGBM GPU availability depends on OpenCL and the installed wheel. Running GPU may change predictions relative to earlier CPU members; `quan_cat_gpu_v1` and `quan_lgb_gpu_v1` have distinct names.

**Inputs (Add Input):** competition `playground-series-s6e10`, dataset `quannh206/fold-data`, and the CS116 notebook output with `cs116-kaggle-s6e10/src/data.py`. Turn on Kaggle Internet if a package must be installed.

**How to persist:** Set accelerator **GPU T4×2** → **Save Version → Save & Run All**. This launches a fresh run top-to-bottom; do NOT run manually for hours and then select Save & Run All (it trains again). Upon successful completion: **Versions → Output Files**, check `/kaggle/working/S6E10_GPU_members.zip`, download it. A full run must finish within Kaggle's 12-hour limit. Progress `.npy` is saved after each fold in `/kaggle/working/.../progress`, but it becomes durable across sessions only if captured in a saved output version or downloaded.


In [1]:
from pathlib import Path
import os, sys, shutil, random, gc, json, csv, importlib, subprocess
from datetime import datetime, timezone, timedelta
import numpy as np
import pandas as pd

RUN_LGB = True
RUN_CAT = True
RUN_TABM = True

# GPU-first: if LightGBM cannot use OpenCL GPU, fall back visibly to CPU.
LGB_GPU_FALLBACK_CPU = True

KAGGLE_PROJECT = Path('/kaggle/input/notebooks/quannh206/cs116/cs116-kaggle-s6e10')
WORK_PROJECT = Path('/kaggle/working/cs116-kaggle-s6e10')
WORK_SRC = WORK_PROJECT / 'src'
DATA_DIR = WORK_PROJECT / 'data'
OUT_DIR = WORK_SRC / 'oof'
PROGRESS_DIR = WORK_PROJECT / 'progress'
for p in [WORK_SRC, DATA_DIR, OUT_DIR, PROGRESS_DIR]: p.mkdir(parents=True, exist_ok=True)

real = {
    'train.csv': Path('/kaggle/input/competitions/playground-series-s6e10/train.csv'),
    'test.csv': Path('/kaggle/input/competitions/playground-series-s6e10/test.csv'),
    'folds.csv': Path('/kaggle/input/datasets/quannh206/fold-data/folds.csv'),
}
for name, src in real.items():
    assert src.is_file(), f'Missing input: {src}'
    dst = DATA_DIR/name
    if dst.is_symlink():
        dst.unlink()
    elif dst.exists():
        raise FileExistsError(f'Refusing to overwrite existing data file: {dst}')
    dst.symlink_to(src)

assert (KAGGLE_PROJECT/'src'/'data.py').is_file(), 'Add CS116 notebook output as Kaggle Input'
# Copy team source to WRITABLE working before import, to avoid modifying input.
shutil.copytree(KAGGLE_PROJECT/'src', WORK_SRC, dirs_exist_ok=True,
                ignore=shutil.ignore_patterns('__pycache__', '*.pyc', 'oof'))

# Import the team's loader from its writable source copy.
for module in ['data', 'utils', 'config']:
    sys.modules.pop(module, None)
sys.path.insert(0, str(WORK_SRC))
import data as team_data
if hasattr(team_data, 'DATA_DIR'):
    team_data.DATA_DIR = DATA_DIR
train, test = team_data.load_data()

expected_folds = pd.read_csv(real['folds.csv'], usecols=['id','fold'])
assert train.shape == (699635, 24), train.shape
assert test.shape == (299844, 22), test.shape
assert np.array_equal(train['id'].to_numpy(), expected_folds['id'].to_numpy()), 'ID order mismatch'
assert np.array_equal(train['fold'].to_numpy(), expected_folds['fold'].to_numpy()), 'Fold mapping mismatch'
assert train['id'].is_unique and test['id'].is_unique
assert set(train['fold'].unique()) == set(range(5))
assert not train['satisfaction'].isna().any()
assert set(np.unique(train['satisfaction'].astype(int))) == {0,1}

# Use official team save_preds if available, but always also save a verified copy
# to the canonical project working output directory.
team_save_preds = None
if (WORK_SRC/'utils.py').is_file():
    import utils as team_utils
    team_save_preds = getattr(team_utils, 'save_preds', None)
print('Official utils.save_preds available:', callable(team_save_preds))

Y = train['satisfaction'].to_numpy(dtype=np.int64)
FOLDS = train['fold'].to_numpy(dtype=np.int64)
FEATURES = [c for c in test.columns if c != 'id']
CATS = ['Gender', 'Customer Type', 'Type of Travel', 'Class']
NUMS = [c for c in FEATURES if c not in CATS]
assert len(FEATURES) == 21 and len(NUMS) == 17
print('Train:', train.shape, 'Test:', test.shape)
print('Fold sizes:', train['fold'].value_counts().sort_index().to_dict())
print('Features:', len(FEATURES), 'Numerical:',len(NUMS), 'Categorical:',len(CATS))
print('Writable output:', OUT_DIR)
# Preflight dependencies BEFORE expensive model training.
for enabled, module, pip_name in [
    (RUN_LGB, 'lightgbm', 'lightgbm'),
    (RUN_CAT, 'catboost', 'catboost'),
    (RUN_TABM, 'torch', 'torch'),
    (RUN_TABM, 'tabm', 'tabm'),
]:
    if enabled and importlib.util.find_spec(module) is None:
        print(f'Installing {pip_name} (requires Kaggle Internet if not cached)...')
        subprocess.check_call([sys.executable,'-m','pip','install','-q',pip_name])
if RUN_TABM:
    import torch
    assert torch.cuda.is_available(), 'Enable GPU T4 in Kaggle settings for TabM'
    print('GPU preflight:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
    print('GPU count:', torch.cuda.device_count())
    print('TabM device:', f'cuda:{1 if torch.cuda.device_count()>1 else 0}')
if RUN_CAT:
    import torch
    assert torch.cuda.is_available(), 'Enable GPU T4 in Kaggle settings for CatBoost GPU'
print('\nGPU status (nvidia-smi):')
subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.total,memory.used', '--format=csv,noheader'],check=False)

Official utils.save_preds available: True
Train: (699635, 24) Test: (299844, 22)
Fold sizes: {0: 139927, 1: 139927, 2: 139927, 3: 139927, 4: 139927}
Features: 21 Numerical: 17 Categorical: 4
Writable output: /kaggle/working/cs116-kaggle-s6e10/src/oof
Installing tabm (requires Kaggle Internet if not cached)...
GPU preflight: ['Tesla T4', 'Tesla T4']
GPU count: 2
TabM device: cuda:1

GPU status (nvidia-smi):
0, Tesla T4, 15360 MiB, 3 MiB
1, Tesla T4, 15360 MiB, 3 MiB


CompletedProcess(args=['nvidia-smi', '--query-gpu=index,name,memory.total,memory.used', '--format=csv,noheader'], returncode=0)

## Helper functions: alignment, fold progress, official output

Each member saves a **partial checkpoint after every completed fold** inside the Kaggle session, plus its final two `.npy` files. If a cell fails during the same session you can re-run that member to resume; **to preserve between sessions, save a Kaggle Version with Output Files**. The partial checkpoints are *not* final member submissions. Never send a partially completed OOF to the stacking lead.

In [2]:
from sklearn.metrics import roc_auc_score

def seed_all(s):
    random.seed(s)
    np.random.seed(s)
    try:
        import torch
        torch.manual_seed(s)
        if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
    except ImportError:
        pass

def init_progress(name, config_tag):
    paths = {
        'meta': PROGRESS_DIR/f'{name}.json',
        'oof': PROGRESS_DIR/f'{name}_partial_oof.npy',
        'testsum': PROGRESS_DIR/f'{name}_partial_testsum.npy',
    }
    if paths['meta'].exists():
        meta=json.loads(paths['meta'].read_text())
        assert meta['config_tag']==config_tag, f'Configuration changed for {name}; use a new member name'
        oof=np.load(paths['oof']); tsum=np.load(paths['testsum']); done=meta['done']
        assert oof.shape==(len(train),) and tsum.shape==(len(test),)
        return oof,tsum,done
    return np.full(len(train),np.nan,np.float64),np.zeros(len(test),np.float64),[]

def write_progress(name, config_tag, oof, tsum, done):
    np.save(PROGRESS_DIR/f'{name}_partial_oof.npy', oof)
    np.save(PROGRESS_DIR/f'{name}_partial_testsum.npy', tsum)
    (PROGRESS_DIR/f'{name}.json').write_text(json.dumps({'config_tag':config_tag,'done':sorted(done)}))

def save_member(name, oof, test_preds):
    assert oof.shape==(699635,) and test_preds.shape==(299844,)
    assert np.isfinite(oof).all() and np.isfinite(test_preds).all()
    assert np.all((oof>=0)&(oof<=1)) and np.all((test_preds>=0)&(test_preds<=1))
    # Always produce canonical and verified output first.
    np.save(OUT_DIR/f'oof_{name}.npy', oof.astype(np.float64))
    np.save(OUT_DIR/f'test_{name}.npy', test_preds.astype(np.float64))
    score=roc_auc_score(Y, oof)
    if callable(team_save_preds):
        here=Path.cwd()
        try:
            os.chdir(WORK_SRC)  # team blend.py expects src/oof under current directory
            team_save_preds(name, oof, test_preds, Y)
        finally:
            os.chdir(here)
    assert np.allclose(np.load(OUT_DIR/f'oof_{name}.npy'),oof)
    print(f'\nSAVED {name}: OOF AUC={score:.9f} | {OUT_DIR}')
    return score

def record_experiment(name, auc, detail):
    csv_path=WORK_PROJECT/'experiments.csv'
    row=[datetime.now(timezone(timedelta(hours=7))).date().isoformat(),
         'quan',name, f'{auc:.9f}', '',detail]
    header=['ngày','người','tên','CV AUC','LB AUC','ghi chú ngắn']
    old=[]
    if csv_path.exists():
        with open(csv_path,newline='',encoding='utf-8') as f:
            allrows=list(csv.reader(f))
        if allrows and allrows[0]==header: old=allrows[1:]
    old=[r for r in old if len(r)<3 or r[2]!=name]
    with open(csv_path,'w',newline='',encoding='utf-8') as f:
        w=csv.writer(f);w.writerow(header);w.writerows(old);w.writerow(row)
    print('Logged:', csv_path)

def fold_scores(oof):
    return [roc_auc_score(Y[FOLDS==k],oof[FOLDS==k]) for k in range(5)]

## Member 1 — LightGBM: GPU first, CPU fallback

A small OpenCL GPU training preflight runs BEFORE expensive 5-fold CV. If the GPU path is unavailable, the notebook prints the exception and uses CPU with member name `quan_lgb_v1`; otherwise it uses `quan_lgb_gpu_v1`. Do not assume the GPU accelerator automatically gives LightGBM GPU support. Always use the shared folds and train-only ES.


In [3]:
if RUN_LGB:
    import lightgbm as lgb
    from sklearn.model_selection import train_test_split
    # Decide which device can *actually* train before starting 5 folds.
    LGB_DEVICE='gpu'
    BASE_TAG='lgb-v1-lr0.05-leaves31-child40-lambda1-maxbin127-es100-4000'
    x=train[FEATURES].copy();xt=test[FEATURES].copy()
    for col in CATS:
        vocab=pd.Index(pd.concat([x[col],xt[col]],ignore_index=True).dropna().unique())
        cat_type=pd.CategoricalDtype(categories=vocab)
        x[col]=x[col].astype(cat_type);xt[col]=xt[col].astype(cat_type)

    def make_lgb(n,seed):
        return lgb.LGBMClassifier(objective='binary',metric='auc',
               n_estimators=n,learning_rate=.05,num_leaves=31,
               min_child_samples=40,reg_lambda=1.0,max_bin=127,
               random_state=seed,n_jobs=-1,verbosity=-1,
               device_type=LGB_DEVICE)

    # Smoke-test LightGBM OpenCL GPU. Kaggle driver/OpenCL availability varies.
    try:
        print('Checking LightGBM OpenCL GPU with 5 trees...')
        probe=make_lgb(5,42)
        probe.fit(x.iloc[:20000],Y[:20000])
        del probe
        print('LightGBM GPU check OK — training on GPU')
    except Exception as e:
        if not LGB_GPU_FALLBACK_CPU:
            raise RuntimeError('LightGBM GPU unavailable; set fallback=True or fix OpenCL') from e
        LGB_DEVICE='cpu'
        print('WARNING: LightGBM GPU NOT available. Falling back to CPU.')
        print(type(e).__name__+':',str(e)[:900])
    NAME='quan_lgb_gpu_v1' if LGB_DEVICE=='gpu' else 'quan_lgb_v1'
    CONFIG=BASE_TAG+'-'+LGB_DEVICE
    print('LightGBM running on:', LGB_DEVICE, '| member:',NAME)
    oof,tsum,done=init_progress(NAME,CONFIG)
    for k in range(5):
        if k in done:
            print(f'LightGBM fold {k} already completed; skipping'); continue
        tr=np.flatnonzero(FOLDS!=k);va=np.flatnonzero(FOLDS==k)
        fit,es=train_test_split(tr,test_size=.1,stratify=Y[tr],random_state=42+k)
        selector=make_lgb(4000,42+k)
        selector.fit(x.iloc[fit],Y[fit],eval_set=[(x.iloc[es],Y[es])],
                     callbacks=[lgb.early_stopping(100,verbose=False)])
        best=max(1,int(selector.best_iteration_))
        del selector
        model=make_lgb(best,42+k)
        model.fit(x.iloc[tr],Y[tr])
        oof[va]=model.predict_proba(x.iloc[va])[:,1]
        tsum+=model.predict_proba(xt)[:,1]
        done.append(k); write_progress(NAME,CONFIG,oof,tsum,done)
        print(f'LightGBM fold {k} AUC: {roc_auc_score(Y[va],oof[va]):.6f} | best trees={best}')
        del model;gc.collect()
    assert len(done)==5
    score=save_member(NAME,oof,tsum/5)
    record_experiment(NAME,score,'21 raw features; internal 10% ES; refit 4 folds; 5-fold shared OOF')
    print('LightGBM fold scores:',fold_scores(oof))
    del x,xt,oof,tsum;gc.collect()

Checking LightGBM OpenCL GPU with 5 trees...


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


LightGBM GPU check OK — training on GPU
LightGBM running on: gpu | member: quan_lgb_gpu_v1
LightGBM fold 0 AUC: 0.958177 | best trees=554
LightGBM fold 1 AUC: 0.957013 | best trees=646
LightGBM fold 2 AUC: 0.959056 | best trees=1017
LightGBM fold 3 AUC: 0.958225 | best trees=524
LightGBM fold 4 AUC: 0.958621 | best trees=1191
Đã lưu oof_quan_lgb_gpu_v1.npy, test_quan_lgb_gpu_v1.npy | CV AUC = 0.958214

SAVED quan_lgb_gpu_v1: OOF AUC=0.958214447 | /kaggle/working/cs116-kaggle-s6e10/src/oof
Logged: /kaggle/working/cs116-kaggle-s6e10/experiments.csv
LightGBM fold scores: [np.float64(0.9581770659194715), np.float64(0.9570128913689266), np.float64(0.9590562366155255), np.float64(0.9582245209594221), np.float64(0.9586213265885216)]


## Member 2 — CatBoost GPU (`quan_cat_gpu_v1`)

CatBoost now uses `task_type='GPU', devices='0'`, not CPU. The GPU algorithm is not bit-identical to the earlier CPU baseline, so its OOF is kept as a separate member. Internal 10% ES determines tree count and the 4 outer-training folds are used for refit. The model NEVER selects trees based on the held-out outer fold.


In [4]:
if RUN_CAT:
    from catboost import CatBoostClassifier
    from sklearn.model_selection import train_test_split
    NAME='quan_cat_gpu_v1'
    CONFIG='cat-gpu-v1-depth7-lr0.05-l2=5-es120-max2500-dev0'
    x=train[FEATURES].copy();xt=test[FEATURES].copy()
    for col in CATS:
        x[col]=x[col].fillna('__MISSING__').astype(str)
        xt[col]=xt[col].fillna('__MISSING__').astype(str)
    def make_cat(n,seed):
        return CatBoostClassifier(iterations=n,depth=7,learning_rate=.05,
                 l2_leaf_reg=5,loss_function='Logloss',eval_metric='AUC',
                 cat_features=CATS,random_seed=seed,task_type='GPU',devices='0',
                 gpu_ram_part=0.75,thread_count=-1,allow_writing_files=False)

    oof,tsum,done=init_progress(NAME,CONFIG)
    for k in range(5):
        if k in done:
            print(f'CatBoost fold {k} already completed; skipping'); continue
        tr=np.flatnonzero(FOLDS!=k);va=np.flatnonzero(FOLDS==k)
        fit,es=train_test_split(tr,test_size=.1,stratify=Y[tr],random_state=42+k)
        selector=make_cat(2500,42+k)
        selector.fit(x.iloc[fit],Y[fit],eval_set=(x.iloc[es],Y[es]),
                     early_stopping_rounds=120,use_best_model=True,verbose=250)
        best=int(selector.best_iteration_)+1
        del selector;gc.collect()
        model=make_cat(best,42+k)
        model.fit(x.iloc[tr],Y[tr],verbose=250)
        oof[va]=model.predict_proba(x.iloc[va])[:,1]
        tsum+=model.predict_proba(xt)[:,1]
        done.append(k);write_progress(NAME,CONFIG,oof,tsum,done)
        print(f'CatBoost fold {k} AUC: {roc_auc_score(Y[va],oof[va]):.6f} | best trees={best}')
        del model;gc.collect()
    assert len(done)==5
    score=save_member(NAME,oof,tsum/5)
    record_experiment(NAME,score,'CatBoost GPU 0; native categories; internal 10% ES; refit 4 folds; 5-fold shared OOF')
    print('CatBoost fold scores:',fold_scores(oof))
    del x,xt,oof,tsum;gc.collect()

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9389570	best: 0.9389570 (0)	total: 140ms	remaining: 5m 49s
250:	test: 0.9569404	best: 0.9569404 (250)	total: 8.38s	remaining: 1m 15s
500:	test: 0.9577508	best: 0.9577508 (500)	total: 16.5s	remaining: 1m 5s
750:	test: 0.9580404	best: 0.9580435 (742)	total: 24.8s	remaining: 57.8s
1000:	test: 0.9582125	best: 0.9582146 (998)	total: 33s	remaining: 49.4s
1250:	test: 0.9582432	best: 0.9582770 (1130)	total: 41.3s	remaining: 41.2s
bestTest = 0.9582770467
bestIteration = 1130
Shrink model to first 1131 iterations.


Default metric period is 5 because AUC is/are not implemented for GPU


0:	total: 34.9ms	remaining: 39.4s
250:	total: 8.71s	remaining: 30.5s
500:	total: 17.5s	remaining: 22.1s
750:	total: 26.3s	remaining: 13.3s
1000:	total: 35.1s	remaining: 4.56s
1130:	total: 39.7s	remaining: 0us
CatBoost fold 0 AUC: 0.958323 | best trees=1131


Default metric period is 5 because AUC is/are not implemented for GPU


...

0:	total: 35.9ms	remaining: 50.2s
250:	total: 8.78s	remaining: 40.2s
500:	total: 17.4s	remaining: 31.3s
750:	total: 26s	remaining: 22.5s
1000:	total: 34.7s	remaining: 13.8s
1250:	total: 43.3s	remaining: 5.16s
1399:	total: 48.5s	remaining: 0us
CatBoost fold 3 AUC: 0.958327 | best trees=1400


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9402491	best: 0.9402491 (0)	total: 52.4ms	remaining: 2m 10s
250:	test: 0.9584542	best: 0.9584569 (249)	total: 8.39s	remaining: 1m 15s
500:	test: 0.9591708	best: 0.9591718 (498)	total: 16.6s	remaining: 1m 6s
750:	test: 0.9594565	best: 0.9594570 (749)	total: 24.9s	remaining: 58s
1000:	test: 0.9595871	best: 0.9595882 (960)	total: 33.2s	remaining: 49.7s
1250:	test: 0.9597068	best: 0.9597113 (1225)	total: 41.5s	remaining: 41.4s
1500:	test: 0.9597979	best: 0.9597979 (1500)	total: 49.8s	remaining: 33.1s
1750:	test: 0.9598572	best: 0.9598623 (1743)	total: 58s	remaining: 24.8s
bestTest = 0.9598835111
bestIteration = 1850
Shrink model to first 1851 iterations.


Default metric period is 5 because AUC is/are not implemented for GPU


0:	total: 35ms	remaining: 1m 4s
250:	total: 8.71s	remaining: 55.5s
500:	total: 17.3s	remaining: 46.7s
750:	total: 25.9s	remaining: 38s
1000:	total: 34.5s	remaining: 29.3s
1250:	total: 43.1s	remaining: 20.7s
1500:	total: 51.6s	remaining: 12s
1750:	total: 1m	remaining: 3.44s
1850:	total: 1m 3s	remaining: 0us
CatBoost fold 4 AUC: 0.958635 | best trees=1851
Đã lưu oof_quan_cat_gpu_v1.npy, test_quan_cat_gpu_v1.npy | CV AUC = 0.958321

SAVED quan_cat_gpu_v1: OOF AUC=0.958321005 | /kaggle/working/cs116-kaggle-s6e10/src/oof
Logged: /kaggle/working/cs116-kaggle-s6e10/experiments.csv
CatBoost fold scores: [np.float64(0.9583232537408066), np.float64(0.9571195870560172), np.float64(0.9592282325773149), np.float64(0.9583271436436105), np.float64(0.9586351595476499)]


## Member 3 — TabM (`quan_tabm_v1`)

Uses 17 numeric features and 4 categorical columns. The numeric imputer and scaler are fit **only on the permitted training subset** in each phase; vocabulary uses feature values only, never targets. TabM trains each branch with BCE-with-logits and averages **probabilities** at inference. For each outer fold: select epoch on an internal 10% holdout, refit from scratch on all four outer-training folds, score the held-out fold; average the five models' test probabilities.

The TabM configuration follows the working pilot: `k=16`, `n_blocks=2`, `d_block=128`, `AdamW(lr=0.002, weight_decay=0.0003)`, `batch_size=1024`, `max_epochs=60`, `patience=10`. Uses `cuda:1` for T4×2 or `cuda:0` for a single GPU.

In [5]:
if RUN_TABM:
    import torch
    import torch.nn.functional as F
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler
    from sklearn.impute import SimpleImputer
    try:
        from tabm import TabM
    except ImportError:
        print('Installing TabM. Kaggle Internet must be enabled if not already available.')
        subprocess.check_call([sys.executable,'-m','pip','install','-q','tabm'])
        from tabm import TabM

    assert torch.cuda.is_available(), 'Enable GPU T4 in Notebook Settings for full TabM run'
    TABM_GPU=1 if torch.cuda.device_count()>1 else 0
    device=torch.device(f'cuda:{TABM_GPU}')
    print('TabM GPU:', TABM_GPU, torch.cuda.get_device_name(TABM_GPU))

    NAME='quan_tabm_v1'
    CONFIG='tabm-v1-block2-d128-k16-lr0.002-wd0.0003-batch1024-max60-patience10'
    N_EPOCHS=60; PATIENCE=10;BATCH=1024

    raw_num=train[NUMS].to_numpy(np.float32)
    test_num=test[NUMS].to_numpy(np.float32)
    x_cat=[];t_cat=[];card=[]
    for col in CATS:
        v=pd.concat([train[col],test[col]],ignore_index=True).fillna('__MISSING__').astype(str)
        category=pd.CategoricalDtype(categories=pd.Index(v.unique()))
        tr_codes=train[col].fillna('__MISSING__').astype(str).astype(category).cat.codes.to_numpy(np.int64)
        te_codes=test[col].fillna('__MISSING__').astype(str).astype(category).cat.codes.to_numpy(np.int64)
        assert (tr_codes>=0).all() and (te_codes>=0).all()
        x_cat.append(tr_codes);t_cat.append(te_codes);card.append(len(category.categories))
    X_CAT=np.column_stack(x_cat)
    T_CAT=np.column_stack(t_cat)

    def prep(inner_tr, inner_val, outer_test=False):
        imp=SimpleImputer(strategy='median')
        z=StandardScaler()
        fitted=imp.fit_transform(raw_num[inner_tr])
        xtr=z.fit_transform(fitted).astype(np.float32)
        if outer_test:
            evalraw=test_num
        else:
            evalraw=raw_num[inner_val]
        xe=z.transform(imp.transform(evalraw)).astype(np.float32)
        assert np.isfinite(xtr).all() and np.isfinite(xe).all()
        return xtr,xe

    def make_tabm():
        return TabM.make(n_num_features=len(NUMS),cat_cardinalities=card,
                         d_out=1,n_blocks=2,d_block=128,k=16).to(device)

    def tensor_num(a): return torch.as_tensor(np.ascontiguousarray(a),dtype=torch.float32,device=device)
    def tensor_cat(a): return torch.as_tensor(np.ascontiguousarray(a),dtype=torch.long,device=device)

    def train_one_epoch(model,optim,num,cat,y_t):
        model.train();order=torch.randperm(len(y_t),device=device)
        loss_sum=0.
        for s in range(0,len(order),BATCH):
            b=order[s:s+BATCH]
            logits=model(num[b],cat[b]).squeeze(-1)
            yt=y_t[b].unsqueeze(1).expand_as(logits)
            loss=F.binary_cross_entropy_with_logits(logits,yt)
            optim.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.)
            optim.step();loss_sum+=loss.item()*len(b)
        return loss_sum/len(y_t)

    @torch.inference_mode()
    def predict(model,num,cat,batch=4096):
        model.eval();p=[]
        for i in range(0,len(num),batch):
            z=model(num[i:i+batch],cat[i:i+batch]).squeeze(-1)
            p.append(torch.sigmoid(z).mean(dim=1).cpu().numpy())
        return np.concatenate(p)

    oof,tsum,done=init_progress(NAME,CONFIG)
    for k in range(5):
        if k in done:
            print(f'TabM fold {k} already completed; skipping');continue
        tr=np.flatnonzero(FOLDS!=k);va=np.flatnonzero(FOLDS==k)
        fit,es=train_test_split(tr,test_size=.1,stratify=Y[tr],random_state=42+k)
        seed_all(42+k)
        nfit,nes=prep(fit,es)
        a,b=tensor_num(nfit),tensor_num(nes)
        ca,cb=tensor_cat(X_CAT[fit]),tensor_cat(X_CAT[es])
        yy=tensor_num(Y[fit].astype(np.float32))
        selector=make_tabm()
        optim=torch.optim.AdamW(selector.parameters(),lr=.002,weight_decay=.0003)
        best_auc=-1.;best_epoch=0;bad=0
        for epoch in range(1,N_EPOCHS+1):
            loss=train_one_epoch(selector,optim,a,ca,yy)
            pred=predict(selector,b,cb)
            score=roc_auc_score(Y[es],pred)
            print(f'TabM fold {k} ES epoch {epoch:02d} | loss {loss:.5f} | AUC {score:.6f}')
            if score>best_auc+1e-5:
                best_auc=score;best_epoch=epoch;bad=0
            else:
                bad+=1
            if bad>=PATIENCE:break
        print(f'TabM fold {k} internal ES best epoch: {best_epoch} (AUC={best_auc:.6f})')
        del selector,optim,a,b,ca,cb,yy,nfit,nes,pred
        gc.collect();torch.cuda.empty_cache()

        # Refit the preprocessors and the model, starting fresh, on all 4 training folds.
        ntr,nva=prep(tr,va)
        _,nte=prep(tr,None,outer_test=True)
        a=tensor_num(ntr);b=tensor_num(nva);c=tensor_num(nte)
        ca=tensor_cat(X_CAT[tr]);cb=tensor_cat(X_CAT[va]);cc=tensor_cat(T_CAT)
        yy=tensor_num(Y[tr].astype(np.float32))
        seed_all(42+k)
        model=make_tabm();optim=torch.optim.AdamW(model.parameters(),lr=.002,weight_decay=.0003)
        for epoch in range(1,best_epoch+1):
            loss=train_one_epoch(model,optim,a,ca,yy)
            if epoch==1 or epoch%5==0 or epoch==best_epoch:
                print(f'TabM fold {k} refit epoch {epoch:02d}/{best_epoch} | loss {loss:.5f}')
        oof[va]=predict(model,b,cb)
        tsum+=predict(model,c,cc)
        done.append(k);write_progress(NAME,CONFIG,oof,tsum,done)
        print(f'TabM fold {k} AUC: {roc_auc_score(Y[va],oof[va]):.6f}')
        del model,optim,a,b,c,ca,cb,cc,yy,ntr,nva,nte
        gc.collect();torch.cuda.empty_cache()
    assert len(done)==5
    score=save_member(NAME,oof,tsum/5)
    record_experiment(NAME,score,'TabM K16 2x128; fold-only preprocessing; internal 10% ES; fold refit; 5-fold OOF')
    print('TabM fold scores:',fold_scores(oof))
    del oof,tsum;gc.collect();torch.cuda.empty_cache()

TabM GPU: 1 Tesla T4
TabM fold 0 ES epoch 01 | loss 0.29262 | AUC 0.952225
TabM fold 0 ES epoch 02 | loss 0.25344 | AUC 0.954600
TabM fold 0 ES epoch 03 | loss 0.24697 | AUC 0.955499
TabM fold 0 ES epoch 04 | loss 0.24393 | AUC 0.955275
TabM fold 0 ES epoch 05 | loss 0.24231 | AUC 0.956372
TabM fold 0 ES epoch 06 | loss 0.24136 | AUC 0.956285
TabM fold 0 ES epoch 07 | loss 0.24028 | AUC 0.956195
TabM fold 0 ES epoch 08 | loss 0.23931 | AUC 0.956533
TabM fold 0 ES epoch 09 | loss 0.23856 | AUC 0.956895
TabM fold 0 ES epoch 10 | loss 0.23803 | AUC 0.957248
TabM fold 0 ES epoch 11 | loss 0.23749 | AUC 0.956923
TabM fold 0 ES epoch 12 | loss 0.23716 | AUC 0.956717
TabM fold 0 ES epoch 13 | loss 0.23673 | AUC 0.957348
TabM fold 0 ES epoch 14 | loss 0.23644 | AUC 0.957409
TabM fold 0 ES epoch 15 | loss 0.23623 | AUC 0.957199
TabM fold 0 ES epoch 16 | loss 0.23594 | AUC 0.957005
TabM fold 0 ES epoch 17 | loss 0.23569 | AUC 0.956881
TabM fold 0 ES epoch 18 | loss 0.23556 | AUC 0.957145
TabM fo

## Final verification & **Save Version** output

Verifies each completed independent member, writes `S6E10_GPU_members.zip` into `/kaggle/working`, and records its files. No blending or submitting. For durability choose **Save Version → Save & Run All** and let that version finish. Check **Versions → Output Files** for `S6E10_GPU_members.zip`; also download the ZIP as an external backup. If you stop a run before completion, files may not be recoverable from a saved version.


In [6]:
from zipfile import ZipFile, ZIP_DEFLATED
from sklearn.metrics import roc_auc_score

MEMBERS=[]
if RUN_LGB: MEMBERS.append('quan_lgb_gpu_v1' if LGB_DEVICE=='gpu' else 'quan_lgb_v1')
if RUN_CAT: MEMBERS.append('quan_cat_gpu_v1')
if RUN_TABM: MEMBERS.append('quan_tabm_v1')
archive=Path('/kaggle/working/S6E10_GPU_members.zip')
files=[]
for name in MEMBERS:
    op=OUT_DIR/f'oof_{name}.npy';tp=OUT_DIR/f'test_{name}.npy'
    if op.exists() and tp.exists():
        a=np.load(op);b=np.load(tp)
        assert a.shape==(699635,) and b.shape==(299844,)
        assert np.isfinite(a).all() and np.isfinite(b).all()
        assert np.all((0<=a)&(a<=1)) and np.all((0<=b)&(b<=1))
        print(f'{name}: OOF AUC={roc_auc_score(Y,a):.9f} | predictions valid')
        files.extend([op,tp])
    else:
        raise RuntimeError(f'Member {name} is INCOMPLETE — refusing to mark run successful')
log=WORK_PROJECT/'experiments.csv'
if log.exists(): files.append(log)
with ZipFile(archive,'w',ZIP_DEFLATED,compresslevel=6) as zipf:
    for f in files: zipf.write(f,arcname=f.name)
print('\nFinal archive:',archive,'size MB:',round(archive.stat().st_size/1024**2,2))
print('FILES:',[f.name for f in files])
print('NEXT: Save Kaggle Version with Output Files, confirm files exist in Version -> Output Files, and download ZIP to keep a backup.')
# Small provenance log for team handoff.
run_info={'members': MEMBERS,'lgb_device': globals().get('LGB_DEVICE','not_run'),'cat_device': 'gpu:0' if RUN_CAT else 'not_run','tabm_device':str(device) if RUN_TABM else 'not_run', 'source_folds':str(real['folds.csv'])}
manifest=Path('/kaggle/working/S6E10_GPU_manifest.json')
manifest.write_text(json.dumps(run_info,indent=2),encoding='utf-8')
print('GPU manifest:',manifest,run_info)


quan_lgb_gpu_v1: OOF AUC=0.958214447 | predictions valid
quan_cat_gpu_v1: OOF AUC=0.958321005 | predictions valid
quan_tabm_v1: OOF AUC=0.957589484 | predictions valid

Final archive: /kaggle/working/S6E10_GPU_members.zip size MB: 18.33
FILES: ['oof_quan_lgb_gpu_v1.npy', 'test_quan_lgb_gpu_v1.npy', 'oof_quan_cat_gpu_v1.npy', 'test_quan_cat_gpu_v1.npy', 'oof_quan_tabm_v1.npy', 'test_quan_tabm_v1.npy', 'experiments.csv']
NEXT: Save Kaggle Version with Output Files, confirm files exist in Version -> Output Files, and download ZIP to keep a backup.
GPU manifest: /kaggle/working/S6E10_GPU_manifest.json {'members': ['quan_lgb_gpu_v1', 'quan_cat_gpu_v1', 'quan_tabm_v1'], 'lgb_device': 'gpu', 'cat_device': 'gpu:0', 'tabm_device': 'cuda:1', 'source_folds': '/kaggle/input/datasets/quannh206/fold-data/folds.csv'}
